# Python objects from Prolog

`OOPLib` lets Prolog code create Python objects, call their methods and read their attributes.
Objects are referred to by atoms (references such as `'<object:list#...>'`) and aliases such as
`$math`; `X := Receiver.method(Args)` calls a method and binds its result.

See also: [Use Python objects from Prolog](https://tuprolog.github.io/2ppy/how-to/use-python-objects-from-prolog/),
[OOPLib reference](https://tuprolog.github.io/2ppy/reference/ooplib/).

In [1]:
from tuprolog.solve.libs import OOPLib
from tuprolog.solve.prolog import prolog_solver

solver = prolog_solver(libraries=OOPLib)
list(solver.query("X := $math.sqrt(2), Y := $math.floor(X)"))

[{'X': 1.4142135623730951, 'Y': 1}]

Objects are created with `new_object/3`, and stay referenced across goals of the same query:

In [2]:
list(
    solver.query("new_object($list, [], L), L.append(c), L.append(a), L.sort, list_items(L, Items)")
)

[{'L': '<object:list#7f0587d60240>', 'Items': ['a', 'c']}]

Keyword arguments are written `Key = Value`; `as` converts values by calling types:

In [3]:
list(solver.query("J := $json.dumps([1, 2, 3], separators = [',', ':']), N := '42' as $int"))

[{'J': '[1,2,3]', 'N': 42}]

## Custom aliases and safety

By default, only side-effect free standard modules are reachable, and private attributes are not.
`oop_lib` builds a library with other limits, or with aliases for your own classes:

In [4]:
from dataclasses import dataclass

from tuprolog.solve.libs import oop_lib


@dataclass
class Point:
    x: float
    y: float

    def norm(self):
        return (self.x**2 + self.y**2) ** 0.5


solver = prolog_solver(libraries=oop_lib(aliases={"point": Point}))
list(solver.query("new_object($point, [3, 4], P), N := P.norm, X := P.x"))

[{'P': '<object:Point#7f0587d5e1b0>', 'N': 5.0, 'X': 3}]